# 📊 SQL 2 – Daten kombinieren und analysieren

## 🎯 Lernziele
Am Ende dieses Labs kannst du:
- Ergebnismengen mit **UNION ALL**, **UNION**, **MINUS** (EXCEPT) und **INTERSECT** stapeln und vergleichen
- Lange Abfragen mit **WITH … AS** (Common Table Expressions) in gut lesbare Bausteine gliedern
- Daten mit **Window-Funktionen** und **PARTITION BY** analysieren – Rankings, laufende Summen und Vergleiche mit dem Vormonat
- Zwischensummen und Gesamtsummen mit **ROLLUP**, **CUBE** und **GROUPING()** bilden
- Hierarchien wie ein Organigramm mit **WITH RECURSIVE** durchlaufen

## Einführung

**Warum dieses Lab?**

In SQL 1 hast du Tabellen mit Joins **nebeneinander** verknüpft. Für viele Aufgaben im Gold Layer brauchst du zwei weitere Fähigkeiten:

- Ergebnismengen **untereinander** kombinieren – z. B. Café-Bons und Online-Bestellungen zu einer Verkaufstabelle (`UNION ALL`) oder zwei Listen vergleichen (`MINUS`, `INTERSECT`)
- **Zeilen im Verhältnis zu anderen Zeilen analysieren** – z. B. „der vorherige Preis dieses Produkts“, „die erste Bestellung dieses Kunden“, „dieser Monat im Vergleich zum Vormonat“. Genau dafür gibt es Window-Funktionen.

Außerdem lernst du, wie du lange Abfragen mit `WITH` lesbar hältst, wie du Zwischensummen wie in einem Bericht erzeugst und wie du mit Hierarchien unbekannter Tiefe umgehst.

## Anleitung

Bevor du startest, prüfe bitte:
- Du hast **SQL 1 – Filtern, Aggregieren und Verknüpfen** abgeschlossen und **Lab 0** hat das Schema `sql_sandbox` angelegt.
- Dieses Notebook ist mit **Serverless** Compute oder einem **SQL warehouse** verbunden.

1. Setze den Standard-Katalog erneut (ersetze ihn, falls du in Lab 0 einen anderen Katalog verwendet hast):

In [ ]:
USE CATALOG sunny_bay_germany;

**Schritt 1: Ergebnismengen stapeln mit UNION ALL und UNION**

Mengenoperatoren kombinieren die Ergebnisse zweier Abfragen **vertikal**. Beide Abfragen müssen die **gleiche Anzahl an Spalten** mit kompatiblen Datentypen liefern; die Spaltennamen kommen aus der ersten Abfrage.

- `UNION ALL` hängt einfach alle Zeilen aneinander – schnell, und Duplikate bleiben erhalten.
- `UNION` hängt aneinander und **entfernt danach Duplikate** – langsamer und nur richtig, wenn Duplikate wirklich unerwünscht sind.

1. Führe das Mini-Beispiel aus und vergleiche die beiden Ergebnisse:

In [ ]:
SELECT city FROM VALUES ('Hamburg'), ('Berlin') AS cafe(city)
UNION ALL
SELECT city FROM VALUES ('Berlin'), ('Leipzig') AS online(city);

SELECT city FROM VALUES ('Hamburg'), ('Berlin') AS cafe(city)
UNION
SELECT city FROM VALUES ('Berlin'), ('Leipzig') AS online(city);

2. Sunny Bay hat zwei Verkaufssysteme. Zähle die Rohbestellungen beider Systeme in einem Ergebnis:

In [ ]:
SELECT 'POS (cafés)' AS source_system, COUNT(*) AS raw_orders
FROM bronze.pos_transactions
UNION ALL
SELECT 'Webshop', COUNT(*)
FROM bronze.web_orders;

> **📝 Hinweis:** Die Silver-Tabelle `silver.sales_order` wird genau so aufgebaut: Die POS-Bons und die Webshop-Bestellungen werden zuerst in dieselbe Form gebracht (gleiche Spalten, gleiche Bedeutung, gleiche Codes) und dann mit `UNION ALL` gestapelt. Öffne `bundle/src/transformations/silver/sales_order.sql` in deinem Git-Ordner, um es dir anzusehen.

3. 🏋️ **Du bist dran:** Erstelle eine Liste aller **Städte**, in denen Sunny Bay Kunden hat – aus dem CRM (`bronze.crm_customers.city`) und aus dem Onlineshop (`bronze.web_accounts.city`). Jede Stadt soll nur einmal vorkommen, alphabetisch sortiert.

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
SELECT city FROM bronze.crm_customers
___                              -- beide Listen stapeln – jede Stadt nur einmal
SELECT city FROM bronze.web_accounts
ORDER BY ___;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 1 › Aufgabe 3*.

**Schritt 2: Listen vergleichen mit MINUS und INTERSECT**

- `MINUS` (oder das Synonym `EXCEPT`) liefert die Zeilen der ersten Abfrage, die in der zweiten Abfrage **nicht** vorkommen.
- `INTERSECT` liefert die Zeilen, die in **beiden** Abfragen vorkommen.

Beide entfernen Duplikate und vergleichen **alle** ausgewählten Spalten.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT sku FROM VALUES ('SB-0101'), ('SB-0102'), ('SB-0103') AS master(sku)
MINUS
SELECT sku FROM VALUES ('SB-0102'), ('SB-0103'), ('SB-9999') AS sold(sku);

SELECT sku FROM VALUES ('SB-0101'), ('SB-0102'), ('SB-0103') AS master(sku)
INTERSECT
SELECT sku FROM VALUES ('SB-0102'), ('SB-0103'), ('SB-9999') AS sold(sku);

2. Welche SKUs aus den Produktstammdaten wurden **noch nie** verkauft?

In [ ]:
SELECT sku FROM silver.product
MINUS
SELECT sku FROM silver.sales_order_line;

3. Und umgekehrt: Welche SKUs wurden an den Café-Kassen gescannt, sind in den Produktstammdaten aber **unbekannt**?

In [ ]:
SELECT sku FROM bronze.pos_transaction_lines
EXCEPT
SELECT sku FROM silver.product;

> **✨ Tipp:** `MINUS` / `EXCEPT` in beide Richtungen ist der schnellste **Abgleich** im Data Engineering: Liefern `A MINUS B` und `B MINUS A` beide keine Zeilen, enthalten zwei Tabellen genau dieselben Zeilen.

4. 🏋️ **Du bist dran:** Welche E-Mail-Adressen gibt es **sowohl** im CRM (`bronze.crm_customers`) als auch im Onlineshop (`bronze.web_accounts`)? Denk an die Lektion aus SQL 1: Normalisiere die CRM-E-Mails mit `LOWER(TRIM(...))`. Zähle sie dann.

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
SELECT COUNT(*) AS emails_in_both_systems
FROM (
    SELECT ___ AS email FROM bronze.crm_customers   -- die CRM-E-Mails normalisieren
    ___                                              -- nur Adressen, die in beiden Listen stehen
    SELECT email FROM bronze.web_accounts
) AS both_systems;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 2 › Aufgabe 4*.

**Schritt 3: Abfragen strukturieren mit WITH … AS**

Eine **Common Table Expression (CTE)** gibt einer Unterabfrage einen Namen. Nach `WITH` definierst du einen oder mehrere benannte Bausteine und verwendest sie in der finalen Abfrage wie Tabellen. Mit CTEs lassen sich lange Abfragen von oben nach unten lesen, und ein Baustein kann mehrfach wiederverwendet werden.

1. Führe das Mini-Beispiel aus:

In [ ]:
WITH prices AS (
    SELECT * FROM VALUES ('Latte', 4.20), ('Espresso', 2.60), ('Flat White', 4.30) AS t(drink, price_eur)
),
expensive AS (
    SELECT * FROM prices WHERE price_eur > 4
)
SELECT COUNT(*) AS expensive_drinks FROM expensive;

2. Wie hoch ist der **durchschnittliche Bestellwert** in den Cafés und online? Summiere zuerst jede Bestellung auf und bilde dann pro Kanal den Durchschnitt der Bestellsummen:

In [ ]:
WITH order_totals AS (
    SELECT
        order_id,
        SUM(quantity * unit_price_eur - discount_eur) AS order_value_eur
    FROM silver.sales_order_line
    GROUP BY order_id
)
SELECT
    o.sales_channel,
    COUNT(*)                         AS orders,
    ROUND(AVG(t.order_value_eur), 2) AS avg_order_value_eur
FROM order_totals AS t
INNER JOIN silver.sales_order AS o
    ON t.order_id = o.order_id
GROUP BY o.sales_channel;

3. 🏋️ **Du bist dran:** Die folgende Abfrage findet die Filialen, deren Anzahl an **Samstags**-Bestellungen über dem Durchschnitt aller Filialen liegt. Sie funktioniert, ist aber schwer zu lesen. Schreibe sie mit zwei CTEs neu: `saturday_orders` (Bestellungen pro Filiale) und `average` (der Durchschnitt dieser Zahlen).

In [ ]:
SELECT store_name, saturday_orders
FROM (
    SELECT s.store_name, COUNT(*) AS saturday_orders
    FROM silver.sales_order AS o
    INNER JOIN silver.store AS s ON o.store_id = s.store_id
    WHERE DAYOFWEEK(o.order_date) = 7
    GROUP BY s.store_name
) AS t
WHERE saturday_orders > (
    SELECT AVG(cnt)
    FROM (
        SELECT COUNT(*) AS cnt
        FROM silver.sales_order
        WHERE DAYOFWEEK(order_date) = 7
        GROUP BY store_id
    ) AS x
);

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
WITH saturday_orders AS (
    -- Anzahl Samstags-Bestellungen pro Filiale: Spalten store_name, saturday_orders
    -- (Tipp: Die innere Abfrage der verschachtelten Version oben macht genau das)
    ___
),
average AS (
    -- eine einzige Zeile: der Durchschnitt von saturday_orders
    SELECT ___ AS avg_saturday_orders
    FROM saturday_orders
)
SELECT so.store_name, so.saturday_orders
FROM saturday_orders AS so
CROSS JOIN average AS a          -- hängt die eine Durchschnittszeile an jede Filiale
WHERE ___
ORDER BY so.saturday_orders DESC;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 3 › Aufgabe 3*.

**Schritt 4: Analysieren mit Window-Funktionen und PARTITION BY**

`GROUP BY` fasst Zeilen zu einer Zeile pro Gruppe zusammen. Eine **Window-Funktion** berechnet einen Wert über eine Gruppe von Zeilen, **behält aber jede Zeile**. Die Klausel `OVER (…)` definiert das Fenster:

- `PARTITION BY` – welche Zeilen zusammengehören (wie ein `GROUP BY`, aber ohne Zusammenfassen)
- `ORDER BY` – die Reihenfolge der Zeilen innerhalb jeder Partition (nötig für Rankings, laufende Summen und `LAG` / `LEAD`)

| Funktion | Was sie liefert |
|---|---|
| `ROW_NUMBER()` | 1, 2, 3, … pro Partition |
| `RANK()` | Rang mit Lücken bei Gleichstand (1, 1, 3) |
| `SUM(x) OVER (…)` | eine Summe – oder eine **laufende Summe**, wenn das Fenster ein `ORDER BY` hat |
| `LAG(x)` / `LEAD(x)` | den Wert der **vorherigen** / **nächsten** Zeile |

1. Führe das Mini-Beispiel aus und sieh dir jede Spalte an:

In [ ]:
SELECT
    store,
    month,
    revenue,
    ROW_NUMBER() OVER (PARTITION BY store ORDER BY month)  AS month_no,
    SUM(revenue) OVER (PARTITION BY store)                 AS store_total,
    SUM(revenue) OVER (PARTITION BY store ORDER BY month)  AS running_total,
    LAG(revenue) OVER (PARTITION BY store ORDER BY month)  AS previous_month
FROM VALUES
    ('Hamburg', 1, 100), ('Hamburg', 2, 120), ('Hamburg', 3, 90),
    ('Berlin',  1, 80),  ('Berlin',  2, 95)
    AS t(store, month, revenue)
ORDER BY store, month;

2. Erstelle ein Ranking der Produkte **pro Filiale** nach Umsatz:

In [ ]:
WITH product_revenue AS (
    SELECT s.store_name, p.product_name, SUM(f.gross_revenue_eur) AS revenue_eur
    FROM gold.fact_coffee_sales AS f
    INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
    INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
    GROUP BY s.store_name, p.product_name
)
SELECT
    store_name,
    product_name,
    revenue_eur,
    RANK() OVER (PARTITION BY store_name ORDER BY revenue_eur DESC) AS rank_in_store
FROM product_revenue
ORDER BY store_name, rank_in_store;

3. Vergleiche jeden Monat pro Filiale mit dem **Vormonat** (Wachstum gegenüber dem Vormonat):

In [ ]:
WITH monthly AS (
    SELECT
        s.store_name,
        DATE_TRUNC('MONTH', d.date)  AS month,
        SUM(f.gross_revenue_eur)     AS revenue_eur
    FROM gold.fact_coffee_sales AS f
    INNER JOIN gold.dim_store AS s ON f.store_key = s.store_key
    INNER JOIN gold.dim_date  AS d ON f.date_key = d.date_key
    GROUP BY s.store_name, DATE_TRUNC('MONTH', d.date)
)
SELECT
    store_name,
    month,
    revenue_eur,
    LAG(revenue_eur) OVER (PARTITION BY store_name ORDER BY month) AS previous_month_eur,
    ROUND(100 * (revenue_eur / LAG(revenue_eur) OVER (PARTITION BY store_name ORDER BY month) - 1), 1)
                                                                   AS growth_pct
FROM monthly
ORDER BY store_name, month;

4. Das ERP speichert bei jeder Preisänderung eine neue Zeile, aber nur mit einem `valid_from`-Datum. Mit `LEAD` siehst du das Startdatum des **nächsten** Preises – der Tag davor ist das Ende des aktuellen Preises. Genau so berechnet `silver.product_price` die Spalte `valid_to`:

In [ ]:
SELECT
    product_id,
    valid_from,
    LEAD(valid_from) OVER (PARTITION BY product_id ORDER BY valid_from) AS next_price_from,
    list_price_eur
FROM bronze.erp_product_prices
ORDER BY product_id, valid_from;

5. 🏋️ **Du bist dran:** Finde für jede Kundenkarte die **erste Café-Bestellung** (die Bestellung mit dem frühesten `order_ts`). Nummeriere die Bestellungen pro Karte in einer CTE mit `ROW_NUMBER()` und behalte nur Nummer 1. Zeige Kartennummer, Bestell-ID und Zeitstempel.

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
WITH numbered_orders AS (
    SELECT
        loyalty_card_no,
        order_id,
        order_ts,
        ROW_NUMBER() OVER (PARTITION BY ___ ORDER BY ___) AS order_no
    FROM silver.sales_order
    WHERE ___                    -- nur Bestellungen mit Kundenkarte
)
SELECT loyalty_card_no, order_id, order_ts
FROM numbered_orders
WHERE ___                        -- nur die erste Bestellung jeder Karte behalten
ORDER BY order_ts;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 4 › Aufgabe 5*.

**Schritt 5: Zwischensummen mit ROLLUP, CUBE und GROUPING()**

Berichte brauchen oft Zwischensummen und eine Gesamtsumme. Statt mehrere Abfragen zu schreiben und sie mit `UNION ALL` zu stapeln, kannst du sie von `GROUP BY` erzeugen lassen:

- `GROUP BY ROLLUP (a, b)` – gruppiert nach `(a, b)`, dann Zwischensummen pro `a`, dann die Gesamtsumme. Verwende es für **Hierarchien** wie Jahr → Monat.
- `GROUP BY CUBE (a, b)` – **alle** Kombinationen: `(a, b)`, `(a)`, `(b)` und die Gesamtsumme. Verwende es für unabhängige Dimensionen.
- `GROUPING(a)` liefert `1`, wenn die Zeile eine Zwischensumme über `a` ist (und `a` daher `NULL` ist), sonst `0`.

1. Führe das Mini-Beispiel aus:

In [ ]:
SELECT
    year,
    month,
    SUM(revenue)    AS revenue,
    GROUPING(month) AS is_year_total
FROM VALUES (2025, 11, 100), (2025, 12, 150), (2026, 1, 120) AS t(year, month, revenue)
GROUP BY ROLLUP (year, month)
ORDER BY year NULLS LAST, month NULLS LAST;

2. Umsatz pro Jahr und Monat mit Jahres-Zwischensummen und einer Gesamtsumme:

In [ ]:
SELECT
    d.year,
    d.month,
    SUM(f.gross_revenue_eur) AS revenue_eur,
    CASE
        WHEN GROUPING(d.year) = 1  THEN 'Grand total'
        WHEN GROUPING(d.month) = 1 THEN 'Year total'
        ELSE 'Month'
    END AS row_type
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_date AS d ON f.date_key = d.date_key
GROUP BY ROLLUP (d.year, d.month)
ORDER BY d.year NULLS LAST, d.month NULLS LAST;

3. Bestellungen nach Verkaufskanal **und** Zahlungsart, mit Summen in jede Richtung:

In [ ]:
SELECT
    CASE WHEN GROUPING(sales_channel) = 1 THEN 'All channels' ELSE sales_channel END AS channel,
    CASE WHEN GROUPING(payment_method) = 1 THEN 'All methods' ELSE payment_method END AS method,
    COUNT(DISTINCT order_id) AS orders
FROM gold.fact_coffee_sales
GROUP BY CUBE (sales_channel, payment_method)
ORDER BY channel, method;

> **📝 Hinweis:** Zwischensummen sind ideal für Berichte und Exporte. In einem **Dashboard** brauchst du sie meist nicht: Die Metric View und das Dashboard berechnen Summen bei Bedarf selbst.

4. 🏋️ **Du bist dran:** Zeige den Bruttoumsatz pro **Filiale** und **Produktkategorie** (aus `gold.dim_store` und `gold.dim_product`), mit einer Zwischensumme pro Filiale und einer Gesamtsumme. Beschrifte die Zwischensummen-Zeilen.

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
SELECT
    s.store_name,
    p.product_category,
    SUM(f.gross_revenue_eur) AS revenue_eur,
    CASE
        WHEN GROUPING(___) = 1 THEN 'Grand total'    -- welche Spalte steht in der Gesamtsumme für „alle Werte“?
        WHEN GROUPING(___) = 1 THEN 'Store total'    -- ... und in der Zwischensumme pro Filiale?
        ELSE 'Category'
    END AS row_type
FROM gold.fact_coffee_sales AS f
INNER JOIN gold.dim_store   AS s ON f.store_key = s.store_key
INNER JOIN gold.dim_product AS p ON f.product_key = p.product_key
GROUP BY ___ (___, ___)          -- ROLLUP oder CUBE? Und in welcher Reihenfolge?
ORDER BY s.store_name NULLS LAST, p.product_category NULLS LAST;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 5 › Aufgabe 4*.

**Schritt 6: Hierarchien durchlaufen mit WITH RECURSIVE**

Manche Daten sind ein **Baum**: Jeder Mitarbeitende hat eine Führungskraft, jede Produktkategorie hat eine übergeordnete Kategorie. Mit normalen Joins brauchst du einen Join pro Ebene – aber wie viele Ebenen gibt es? Eine **rekursive CTE** durchläuft einen Baum beliebiger Tiefe. Sie hat immer zwei Teile, die mit `UNION ALL` verbunden werden:

- den **Anker**: die Startzeilen (z. B. die Person ohne Führungskraft)
- den **rekursiven Teil**: verknüpft die Quelltabelle mit der CTE selbst, um die nächste Ebene zu finden – und wiederholt das, bis keine neuen Zeilen mehr gefunden werden

> **📝 Hinweis:** Rekursive CTEs brauchen ein aktuelles SQL warehouse (oder Databricks Runtime 17.0 und höher). Bekommst du einen Syntaxfehler, bitte deine Kursleitung, den Warehouse-Channel zu prüfen.

1. Führe das Mini-Beispiel aus – ein Mini-Organigramm mit drei Ebenen:

In [ ]:
WITH RECURSIVE chart (name, level) AS (
    SELECT name, 1
    FROM VALUES ('Benno', NULL), ('Katrin', 'Benno'), ('Jonas', 'Benno'), ('Mila', 'Katrin') AS t(name, manager)
    WHERE manager IS NULL
    UNION ALL
    SELECT t.name, c.level + 1
    FROM VALUES ('Benno', NULL), ('Katrin', 'Benno'), ('Jonas', 'Benno'), ('Mila', 'Katrin') AS t(name, manager)
    INNER JOIN chart AS c ON t.manager = c.name
)
SELECT * FROM chart ORDER BY level, name;

2. Erstelle aus den HR-Daten das vollständige **Organigramm** der Sunny Bay Roastery, mit der Ebene und dem Berichtsweg jedes Mitarbeitenden:

In [ ]:
WITH RECURSIVE org_chart (employee_id, full_name, job_title, level, reporting_path) AS (
    -- Anker: Der Gründer hat keine Führungskraft
    SELECT employee_id, full_name, job_title, 1, full_name
    FROM silver.employee
    WHERE manager_id IS NULL

    UNION ALL

    -- rekursiver Teil: alle, deren Führungskraft schon im Organigramm ist
    SELECT e.employee_id, e.full_name, e.job_title, oc.level + 1, CONCAT(oc.reporting_path, ' > ', e.full_name)
    FROM silver.employee AS e
    INNER JOIN org_chart AS oc
        ON e.manager_id = oc.employee_id
)
SELECT level, job_title, full_name, reporting_path
FROM org_chart
ORDER BY reporting_path;

3. 🏋️ **Du bist dran:** Erstelle den **Kategoriepfad** jedes Produkts, z. B. `All Products > Drink > Milk Coffee > Cappuccino`. Starte die Rekursion bei der Wurzelkategorie (`parent_category_id IS NULL`) von `silver.product_category`, gehe den Baum hinunter und verknüpfe zum Schluss die Produkte mit ihrer Kategorie (`silver.product.category_id`).

In [ ]:
-- 🏋️ Du bist dran – Startpunkt: ersetze jedes ___ und führe die Zelle aus
WITH RECURSIVE category_path (category_id, path, depth) AS (
    -- ① Anker: bei der Wurzelkategorie starten
    SELECT category_id, category_name, 1
    FROM silver.product_category
    WHERE ___

    UNION ALL

    -- ② rekursiver Schritt: die Kinder der bisher gefundenen Kategorien
    SELECT c.category_id, CONCAT(cp.path, ' > ', ___), cp.depth + 1
    FROM silver.product_category AS c
    INNER JOIN category_path AS cp
        ON ___
)
-- ③ jedes Produkt an den Pfad seiner Kategorie hängen
SELECT
    p.product_name,
    CONCAT(___, ' > ', ___) AS full_path,
    cp.depth + 1            AS depth
FROM silver.product AS p
INNER JOIN category_path AS cp
    ON ___
ORDER BY full_path;

> **💡 Lösung:** Hängst du fest oder bist du fertig? Vergleiche mit dem Notebook **SQL Lösungen** in diesem Ordner → *SQL 2 › Schritt 6 › Aufgabe 3*.

## Wie geht es weiter?

🎉 Super gemacht! Du kannst jetzt Ergebnismengen stapeln und vergleichen, deine Abfragen mit CTEs strukturieren, Zeilen im Kontext mit Window-Funktionen analysieren, Zwischensummen erzeugen und Hierarchien durchlaufen.

In **SQL 3 – Funktionen für den Gold Layer** lernst du die Funktionen kennen, die unordentliche Quelldaten in saubere Gold-Tabellen verwandeln – `EXPLODE`, `QUALIFY`, `COALESCE`, `CASE`/`IFF`, `DECODE`, `MAX_BY`, `FILTER` und `DATEADD` – und zum Abschluss des SQL Track baust du selbst eine komplette Gold-Dimension.